In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import requests
import base64
import json
from typing import Dict, List, Optional

class MetomaticsWeatherModel:
    def __init__(self, username: str, password: str):
        """
        Initialize the Metomatics Weather Forecast Model
        
        Args:
            username: Metomatics API username
            password: Metomatics API password
        """
        self.username = 'ramaila_johannes'
        self.password = 'PHN68Mo5C4Wwh3td3r9m'
        self.base_url = "https://api.meteomatics.com"
        self.historical_data = pd.DataFrame()
        self.model_trained = False
        
    def _get_auth_token(self) -> str:
        """
        Get authentication token for Metomatics API
        """
        credentials = f"{self.username}:{self.password}"
        token = base64.b64encode(credentials.encode()).decode()
        return token
    
    def fetch_historical_data(self, lat: float, lon: float, years_back: int = 10) -> pd.DataFrame:
        """
        Fetch historical weather data from Metomatics API
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=365 * years_back)
        
        # Format dates for API
        start_str = start_date.strftime('%Y-%m-%dT00:00:00Z')
        end_str = end_date.strftime('%Y-%m-%dT23:59:59Z')
        
        # Weather parameters to fetch
        parameters = [
            't_2m:C',           # Temperature at 2m in Celsius
            'relative_humidity_2m:p',  # Relative humidity at 2m in %
            'precip_1h:mm',     # Precipitation in mm
            'wind_speed_10m:ms' # Wind speed at 10m in m/s
        ]
        
        all_data = []
        
        print("Fetching historical data from Metomatics API...")
        
        # Fetch data year by year to avoid API limits
        for year_offset in range(years_back):
            current_start = start_date + timedelta(days=365 * year_offset)
            current_end = current_start + timedelta(days=365)
            
            if current_end > end_date:
                current_end = end_date
                
            current_start_str = current_start.strftime('%Y-%m-%dT00:00:00Z')
            current_end_str = current_end.strftime('%Y-%m-%dT23:59:59Z')
            
            try:
                data = self._fetch_year_data(lat, lon, current_start_str, current_end_str, parameters)
                if data:
                    all_data.extend(data)
                    print(f"Fetched data for {current_start.year}")
            except Exception as e:
                print(f"Error fetching data for {current_start.year}: {e}")
                # Fallback to synthetic data
                all_data.extend(self._create_synthetic_data(current_start, current_end, lat, lon))
        
        if not all_data:
            print("No data received from API, using synthetic data for demonstration")
            all_data = self._create_synthetic_data(start_date, end_date, lat, lon)
        
        return pd.DataFrame(all_data)
    
    def _fetch_year_data(self, lat: float, lon: float, start_date: str, end_date: str, parameters: List[str]) -> List[Dict]:
        """
        Fetch data for a specific year from Metomatics API
        """
        parameters_str = ','.join(parameters)
        url = f"{self.base_url}/{start_date}--{end_date}:P1D/{parameters_str}/{lat},{lon}/json"
        
        headers = {
            'Authorization': f'Basic {self._get_auth_token()}',
            'Content-Type': 'application/json'
        }
        
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            return self._parse_api_response(response.json(), start_date, end_date)
        else:
            print(f"API request failed with status {response.status_code}")
            return []
    
    def _parse_api_response(self, api_data: Dict, start_date: str, end_date: str) -> List[Dict]:
        """
        Parse Metomatics API response into structured data
        """
        parsed_data = []
        
        try:
            if 'data' not in api_data or not api_data['data']:
                return []
            
            first_parameter = api_data['data'][0]
            dates = [datetime.fromisoformat(ts['date'].replace('Z', '+00:00')) for ts in first_parameter['coordinates'][0]['dates']]
            
            temp_data = {}
            for parameter in api_data['data']:
                param_name = parameter['parameter']
                values = parameter['coordinates'][0]['dates']
                
                for i, value_point in enumerate(values):
                    if i >= len(dates):
                        continue
                    date = dates[i]
                    if date not in temp_data:
                        temp_data[date] = {'date': date}
                    
                    temp_data[date][param_name] = value_point['value']
            
            for date, values in temp_data.items():
                parsed_data.append(values)
                
        except Exception as e:
            print(f"Error parsing API response: {e}")
        
        return parsed_data
    
    def _create_synthetic_data(self, start_date: datetime, end_date: datetime, lat: float, lon: float) -> List[Dict]:
        """
        Create synthetic data for demonstration when API is unavailable
        """
        print("Creating synthetic data for demonstration...")
        dates = pd.date_range(start=start_date, end=end_date, freq='D')
        synthetic_data = []
        
        for date in dates:
            day_of_year = date.timetuple().tm_yday
            
            # Base patterns with seasonal variation
            base_temp = 15 + 10 * np.sin(2 * np.pi * (day_of_year - 80) / 365) # Peak around day 172 (June 21)
            temperature = base_temp + np.random.normal(0, 3) # Add some noise
            
            humidity = max(30, min(100, 60 + 20 * np.sin(2 * np.pi * (day_of_year - 100) / 365) + np.random.normal(0, 10))) # Seasonal humidity
            
            precip_prob = 0.3 + 0.2 * np.sin(2 * np.pi * (day_of_year - 300) / 365)
            precipitation = np.random.exponential(2) if np.random.random() < precip_prob else 0
            
            wind_speed = max(0, np.random.exponential(3) + 2 * np.sin(2 * np.pi * (day_of_year - 150) / 365))
            
            synthetic_data.append({
                'date': date,
                't_2m:C': round(temperature, 1),
                'relative_humidity_2m:p': round(humidity, 1),
                'precip_1h:mm': round(precipitation, 1),
                'wind_speed_10m:ms': round(wind_speed, 1),
                'year': date.year,
                'month': date.month,
                'day': date.day
            })
        
        return synthetic_data
    
    def train_model(self, lat: float, lon: float, years_back: int = 5):
        """
        Train the model with historical data from Metomatics API
        """
        print(f"Training model with {years_back} years of historical data...")
        print(f"Location: Latitude {lat}, Longitude {lon}")
        
        self.historical_data = self.fetch_historical_data(lat, lon, years_back)
        
        if self.historical_data.empty:
            raise ValueError("No historical data available for training")
        
        self._standardize_columns()
        self.daily_stats = self._calculate_daily_statistics()
        
        self.model_trained = True
        print(f"Model training completed! Processed {len(self.historical_data)} days of historical data")
    
    def _standardize_columns(self):
        """Standardize column names for easier processing"""
        column_mapping = {
            't_2m:C': 'temperature',
            'relative_humidity_2m:p': 'humidity', 
            'precip_1h:mm': 'precipitation',
            'wind_speed_10m:ms': 'wind_speed'
        }
        
        for old_col, new_col in column_mapping.items():
            if old_col in self.historical_data.columns:
                self.historical_data[new_col] = self.historical_data[old_col]
        
        self.historical_data['year'] = self.historical_data['date'].dt.year
        self.historical_data['month'] = self.historical_data['date'].dt.month
        self.historical_data['day'] = self.historical_data['date'].dt.day
    
    def _calculate_daily_statistics(self) -> Dict:
        """
        Calculate statistics for each day of the year
        """
        daily_stats = {}
        
        for month in range(1, 13):
            for day in range(1, 32):
                try:
                    mask = (self.historical_data['month'] == month) & (self.historical_data['day'] == day)
                    day_data = self.historical_data[mask]
                    
                    if len(day_data) > 0:
                        stats = {
                            'temperature_mean': day_data['temperature'].mean(),
                            'temperature_std': day_data['temperature'].std(),
                            'temperature_min': day_data['temperature'].min(),
                            'temperature_max': day_data['temperature'].max(),
                            'humidity_mean': day_data['humidity'].mean(),
                            'humidity_std': day_data['humidity'].std(),
                            'precipitation_mean': day_data['precipitation'].mean(),
                            'precipitation_std': day_data['precipitation'].std(),
                            'wind_speed_mean': day_data['wind_speed'].mean(),
                            'wind_speed_std': day_data['wind_speed'].std(),
                            'data_points': len(day_data)
                        }
                        daily_stats[(month, day)] = stats
                except:
                    continue
        
        return daily_stats
    
    def predict_weather(self, target_date: str) -> Dict:
        """
        Predict weather for a specific date
        """
        if not self.model_trained:
            raise ValueError("Model must be trained before making predictions")
        
        # Parse target date
        try:
            if '-' in target_date:
                target_dt = datetime.strptime(target_date, '%Y-%m-%d')
            else:
                target_dt = datetime.strptime(target_date, '%d %B %Y')
        except:
            raise ValueError("Invalid date format. Use 'YYYY-MM-DD' or 'DD Month YYYY'")
        
        month = target_dt.month
        day = target_dt.day
        
        if (month, day) not in self.daily_stats:
            raise ValueError(f"No historical data available for {month}/{day}")
        
        stats = self.daily_stats[(month, day)]
        
        # Generate prediction based on historical statistics
        prediction = {
            'date': target_dt.strftime('%Y-%m-%d'),
            'temperature': round(np.random.normal(stats['temperature_mean'], stats['temperature_std']), 1),
            'humidity': round(max(0, min(100, np.random.normal(stats['humidity_mean'], stats['humidity_std']))), 1),
            'precipitation': round(max(0, np.random.normal(stats['precipitation_mean'], stats['precipitation_std'])), 1),
            'wind_speed': round(max(0, np.random.normal(stats['wind_speed_mean'], stats['wind_speed_std'])), 1),
            'confidence': 'high' if stats['data_points'] > 3 else 'medium',
            'historical_based_on': stats['data_points'],
            'historical_stats': {
                'avg_temperature': round(stats['temperature_mean'], 1),
                'avg_humidity': round(stats['humidity_mean'], 1),
                'avg_precipitation': round(stats['precipitation_mean'], 1),
                'avg_wind_speed': round(stats['wind_speed_mean'], 1)
            }
        }
        
        return prediction

def get_user_input():
    """
    Get location and date input from user
    """
    print(" Weather Prediction Model ")
    print("=" * 40)
    
    # Get location input
    print("\nEnter your location:")
    print("1. Use coordinates (latitude and longitude)")
    print("2. Use city name (major cities only)")
    
    choice = input("Choose option (1 or 2): ").strip()
    
    if choice == "1":
        try:
            lat = float(input("Enter latitude (e.g., 40.7128): "))
            lon = float(input("Enter longitude (e.g., -74.0060): "))
            locati2on_name = f"Custom Location ({lat}, {lon})"
        except ValueError:
            print("Invalid coordinates. Using default location (New York).")
            lat, lon, location_name = 40.7128, -74.0060, "New York"
    
    elif choice == "2":
        city = input("Enter city name: ").strip().lower()
        
        # Predefined cities with coordinates
        cities = {}
        
        if city in cities:
            lat, lon, location_name = cities[city]
        else:
            print(f"City '{city}' not in database. Using New York as default.")
            lat, lon, location_name = 40.7128, -74.0060, "New York"
    
    else:
        print("Invalid choice. Using default location (New York).")
        lat, lon, location_name = 40.7128, -74.0060, "New York"
    
    # Get date input
    print("\nEnter the date for weather prediction:")
    print("Format: YYYY-MM-DD (e.g., 2026-01-07) or DD Month YYYY (e.g., 07 January 2026)")
    
    date_input = input("Date: ").strip()
    
    # Validate date is within next 6 months
    try:
        if '-' in date_input:
            target_date = datetime.strptime(date_input, '%Y-%m-%d')
        else:
            target_date = datetime.strptime(date_input, '%d %B %Y')
        
        six_months_later = datetime.now() + timedelta(days=180)
        
        if target_date > six_months_later:
            print("Warning: Date is more than 6 months from now. Prediction may be less accurate.")
        
    except ValueError:
        print("Invalid date format. Using 3 months from today as default.")
        target_date = datetime.now() + timedelta(days=90)
        date_input = target_date.strftime('%Y-%m-%d')
    
    return lat, lon, location_name, date_input

def main():
    """
    Main function to run the weather prediction model with user input
    """
    # Replace with your actual Metomatics credentials
    USERNAME = "ramaila_johannes"
    PASSWORD = "PHN68Mo5C4Wwh3td3r9m"
    
    # For demo purposes, you can use these test credentials or leave as is
    # Note: You need to sign up at https://www.meteomatics.com/en/weather-api/
    
    try:
        # Initialize the model
        model = MetomaticsWeatherModel(username=USERNAME, password=PASSWORD)
        
        # Get user input
        lat, lon, location_name, target_date = get_user_input()
        
        print(f"\n{'='*50}")
        print(f"PROCESSING WEATHER PREDICTION")
        print(f"{'='*50}")
        print(f"Location: {location_name}")
        print(f"Coordinates: {lat}, {lon}")
        print(f"Target Date: {target_date}")
        print(f"{'='*50}")
        
        # Train the model
        print("\nTraining model with historical data...")
        model.train_model(lat=lat, lon=lon, years_back=5)
        
        # Get prediction
        print(f"\nGenerating weather prediction...")
        prediction = model.predict_weather(target_date)
        
        # Display results
        print(f"\n WEATHER PREDICTION FOR {prediction['date']}")
        print(f" Location: {location_name}")
        print(f" Temperature: {prediction['temperature']}°C")
        print(f" Humidity: {prediction['humidity']}%")
        print(f"  Precipitation: {prediction['precipitation']}mm")
        print(f" Wind Speed: {prediction['wind_speed']} m/s")
        print(f" Confidence: {prediction['confidence']}")
        print(f"Based on: {prediction['historical_based_on']} historical data points")
        
        # Show historical averages for comparison
        print(f"\n  HISTORICAL AVERAGES FOR THIS DATE:")
        print(f"   Average Temperature: {prediction['historical_stats']['avg_temperature']}°C")
        print(f"   Average Humidity: {prediction['historical_stats']['avg_humidity']}%")
        print(f"   Average Precipitation: {prediction['historical_stats']['avg_precipitation']}mm")
        print(f"   Average Wind Speed: {prediction['historical_stats']['avg_wind_speed']} m/s")
        
        # Option to get another prediction
        while True:
            another = input("\nWould you like to get another prediction? (y/n): ").strip().lower()
            if another in ['y', 'yes']:
                lat, lon, location_name, target_date = get_user_input()
                prediction = model.predict_weather(target_date)
                
                print(f"\n WEATHER PREDICTION FOR {prediction['date']}")
                print(f"Location: {location_name}")
                print(f" Temperature: {prediction['temperature']}°C")
                print(f"Humidity: {prediction['humidity']}%")
                print(f"Precipitation: {prediction['precipitation']}mm")
                print(f"Wind Speed: {prediction['wind_speed']} m/s")
                print(f"Confidence: {prediction['confidence']}")
                
            elif another in ['n', 'no']:
                print("Thank you for using the Weather Prediction Model!")
                break
            else:
                print("Please enter 'y' or 'n'")
                
    except Exception as e:
        print(f"An error occurred: {e}")
        print("Please check your Metomatics credentials and try again.")
        
if __name__ == "__main__":
    main()

 Weather Prediction Model 

Enter your location:
1. Use coordinates (latitude and longitude)
2. Use city name (major cities only)
City 'new york' not in database. Using New York as default.

Enter the date for weather prediction:
Format: YYYY-MM-DD (e.g., 2026-01-07) or DD Month YYYY (e.g., 07 January 2026)

PROCESSING WEATHER PREDICTION
Location: New York
Coordinates: 40.7128, -74.006
Target Date: 2025-10-30

Training model with historical data...
Training model with 5 years of historical data...
Location: Latitude 40.7128, Longitude -74.006
Fetching historical data from Metomatics API...
Fetched data for 2020
Fetched data for 2021
Fetched data for 2022
Fetched data for 2023
Fetched data for 2024
Model training completed! Processed 1830 days of historical data

Generating weather prediction...

 WEATHER PREDICTION FOR 2025-10-30
 Location: New York
 Temperature: 18.5°C
 Humidity: 79.9%
  Precipitation: 0mm
 Wind Speed: 12.3 m/s
 Confidence: high
Based on: 5 historical data points

  H